# 🛰️ Uydu Telemetri Veri İnceleme (EDA)
## ESA OPS-SAT Anomali Tespiti Projesi

**Amaç:** ESA OPS-SAT uydu telemetri verilerini keşifsel olarak analiz etmek.

### Telemetri Kanalları
| Kanal ID | Sensör Tipi |
|----------|------------|
| CADC0872 | Manyetometre X (I_B_FB_MM_0) |
| CADC0873 | Manyetometre Y (I_B_FB_MM_1) |
| CADC0874 | Manyetometre Z (I_B_FB_MM_2) |
| CADC0884 | Foto Diyot 1 (I_PD1_THETA) |
| CADC0886 | Foto Diyot 2 (I_PD2_THETA) |
| CADC0888 | Foto Diyot 3 (I_PD3_THETA) |
| CADC0890 | Foto Diyot 4 (I_PD4_THETA) |
| CADC0892 | Foto Diyot 5 (I_PD5_THETA) |
| CADC0894 | Foto Diyot 6 (I_PD6_THETA) |

In [ ]:
# Gerekli kutuphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Gorsellestirme ayarlari
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Kanal isimlendirme sozlugu
CHANNEL_NAMES = {
    'CADC0872': 'Manyetometre X', 'CADC0873': 'Manyetometre Y',
    'CADC0874': 'Manyetometre Z', 'CADC0884': 'Foto Diyot 1',
    'CADC0886': 'Foto Diyot 2', 'CADC0888': 'Foto Diyot 3',
    'CADC0890': 'Foto Diyot 4', 'CADC0892': 'Foto Diyot 5',
    'CADC0894': 'Foto Diyot 6'
}
print('✅ Kutuphaneler yuklendi.')

### 1.1 Veri Yuklemesi

In [ ]:
# Veri dosyalarini yukle
segments = pd.read_csv('../data/raw/segments.csv')
dataset = pd.read_csv('../data/raw/dataset.csv')

# Zaman damgasini parse et
segments['timestamp'] = pd.to_datetime(segments['timestamp'])

print(f'📊 Segments: {segments.shape[0]:,} satir x {segments.shape[1]} sutun')
print(f'📊 Dataset:  {dataset.shape[0]:,} satir x {dataset.shape[1]} sutun')

### 1.2 Ilk ve Son Satirlar

In [ ]:
print('=== SEGMENTS - Ilk 5 Satir ===')
display(segments.head())
print('\n=== SEGMENTS - Son 5 Satir ===')
display(segments.tail())

In [ ]:
print('=== DATASET (Ozellik) - Ilk 5 Satir ===')
display(dataset.head())
print('\n=== DATASET - Son 5 Satir ===')
display(dataset.tail())

### 1.3 Sutun ve Veri Tipleri

In [ ]:
print('=== SEGMENTS Sutun Bilgileri ===')
for col in segments.columns:
    print(f'  {col:20s} | {str(segments[col].dtype):10s} | Benzersiz: {segments[col].nunique():>6,} | Bos: {segments[col].isnull().sum()}')
print(f'\n=== DATASET Sutun Bilgileri ===')
for col in dataset.columns:
    print(f'  {col:20s} | {str(dataset[col].dtype):10s} | Benzersiz: {dataset[col].nunique():>6,} | Bos: {dataset[col].isnull().sum()}')

---
## 📊 Bolum 2: Temel Istatistiksel Analiz

In [ ]:
# Segments - sayi degerleri istatistikleri
print('=== Segments - Temel Istatistikler ===')
display(segments.describe())

In [ ]:
# Dataset - ozellik istatistikleri
print('=== Dataset (Ozellikler) - Temel Istatistikler ===')
display(dataset.describe())

### 2.1 Kanal Bazinda Detayli Istatistikler

In [ ]:
# Her kanal icin detayli istatistik
from scipy import stats as scipy_stats

channel_stats = []
for ch in segments['channel'].unique():
    ch_data = segments[segments['channel'] == ch]['value']
    channel_stats.append({
        'Kanal': ch,
        'Sensor': CHANNEL_NAMES.get(ch, ch),
        'Ornek_Sayisi': len(ch_data),
        'Ortalama': ch_data.mean(),
        'Std': ch_data.std(),
        'Min': ch_data.min(),
        'Max': ch_data.max(),
        'Medyan': ch_data.median(),
        'Carpiklik': ch_data.skew(),
        'Basiklik': ch_data.kurtosis(),
        'Q1': ch_data.quantile(0.25),
        'Q3': ch_data.quantile(0.75),
        'IQR': ch_data.quantile(0.75) - ch_data.quantile(0.25)
    })

stats_df = pd.DataFrame(channel_stats).set_index('Kanal')
display(stats_df.style.format('{:.6f}', subset=[c for c in stats_df.columns if stats_df[c].dtype == 'float64'])
    .background_gradient(cmap='YlOrRd', subset=['Carpiklik','Basiklik']))
print('\n📌 Yuksek carpiklik/basiklik degerleri anormal dagilim isaretcisidir.')

---
## 🔍 Bolum 3: Eksik Veri Analizi

In [ ]:
# Eksik deger analizi
print('=== SEGMENTS - Eksik Degerler ===')
missing_seg = segments.isnull().sum()
missing_pct = (missing_seg / len(segments) * 100).round(2)
missing_df = pd.DataFrame({'Eksik': missing_seg, 'Yuzde (%)': missing_pct})
display(missing_df[missing_df['Eksik'] > 0] if missing_df['Eksik'].sum() > 0 else print('Eksik deger yok!'))

print('\n=== DATASET - Eksik Degerler ===')
missing_ds = dataset.isnull().sum()
missing_pct2 = (missing_ds / len(dataset) * 100).round(2)
missing_df2 = pd.DataFrame({'Eksik': missing_ds, 'Yuzde (%)': missing_pct2})
display(missing_df2[missing_df2['Eksik'] > 0] if missing_df2['Eksik'].sum() > 0 else print('Eksik deger yok!'))

### 3.1 Eksik Veri Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Segments eksik veri
sns.heatmap(segments.isnull().astype(int), cbar=True, yticklabels=False, ax=axes[0], cmap='YlOrRd')
axes[0].set_title('Segments - Eksik Veri Haritasi', fontsize=14, fontweight='bold')

# Dataset eksik veri
sns.heatmap(dataset.isnull().astype(int), cbar=True, yticklabels=False, ax=axes[1], cmap='YlOrRd')
axes[1].set_title('Dataset - Eksik Veri Haritasi', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/eksik_veri_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📁 Grafik kaydedildi: reports/figures/eksik_veri_heatmap.png')

### 3.2 Kanal Bazinda Veri Yogunlugu

In [ ]:
# Her kanalda kac veri noktasi var
channel_counts = segments.groupby('channel').size().reset_index(name='count')
channel_counts['sensor'] = channel_counts['channel'].map(CHANNEL_NAMES)

fig = px.bar(channel_counts, x='sensor', y='count', color='count',
             title='📡 Kanal Bazinda Veri Noktasi Sayisi',
             labels={'sensor': 'Sensor', 'count': 'Veri Noktasi'},
             color_continuous_scale='Viridis')
fig.update_layout(template='plotly_dark', height=450)
fig.show()

---
## 📈 Bolum 4: Dagilim Analizi

### 4.1 Histogram ve KDE Grafikleri

In [ ]:
channels = segments['channel'].unique()
n_ch = len(channels)
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, ch in enumerate(channels):
    ch_data = segments[segments['channel'] == ch]['value']
    ax = axes[i]
    ax.hist(ch_data, bins=80, density=True, alpha=0.6, color=f'C{i}', edgecolor='black', linewidth=0.3)
    ch_data.plot.kde(ax=ax, color='red', linewidth=2)
    ax.set_title(f'{CHANNEL_NAMES.get(ch, ch)}\n({ch})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Deger')
    ax.set_ylabel('Yogunluk')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 Telemetri Kanallari - Dagilim Analizi (Histogram + KDE)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/dagilim_histogram_kde.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Box Plot - Aykiri Deger Gorsellestirmesi

In [ ]:
fig = px.box(segments, x='channel', y='value', color='channel',
             title='📦 Kanal Bazinda Box Plot - Aykiri Deger Analizi',
             labels={'channel': 'Kanal', 'value': 'Telemetri Degeri'})
fig.update_layout(template='plotly_dark', height=500, showlegend=False,
                  xaxis_ticktext=[CHANNEL_NAMES.get(c,c) for c in segments['channel'].unique()],
                  xaxis_tickvals=list(segments['channel'].unique()))
fig.show()

### 4.3 Violin Plot - Normal vs Anomali

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, ch in enumerate(channels):
    ch_data = segments[segments['channel'] == ch]
    ax = axes[i]
    parts = ax.violinplot(
        [ch_data[ch_data['anomaly']==0]['value'].values,
         ch_data[ch_data['anomaly']==1]['value'].values],
        showmeans=True, showmedians=True)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Normal', 'Anomali'])
    ax.set_title(f'{CHANNEL_NAMES.get(ch, ch)}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Deger')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('🎻 Normal vs Anomali Dagilimi (Violin Plot)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/violin_normal_anomali.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔗 Bolum 5: Korelasyon Analizi

### 5.1 Dataset Ozellikleri - Pearson Korelasyon Matrisi

In [ ]:
numeric_cols = dataset.select_dtypes(include=[np.number]).columns
corr_pearson = dataset[numeric_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_pearson, dtype=bool))
sns.heatmap(corr_pearson, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Korelasyon Katsayisi'})
ax.set_title('Pearson Korelasyon Matrisi - Dataset Ozellikleri', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/korelasyon_pearson.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Spearman Korelasyon Matrisi

In [ ]:
corr_spearman = dataset[numeric_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))
sns.heatmap(corr_spearman, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Spearman Korelasyon Matrisi', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/korelasyon_spearman.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Interaktif Korelasyon Heatmap

In [ ]:
fig = px.imshow(corr_pearson, text_auto='.2f', color_continuous_scale='RdBu_r',
                title='🔗 Interaktif Pearson Korelasyon Matrisi',
                labels=dict(color='Korelasyon'))
fig.update_layout(template='plotly_dark', height=700, width=800)
fig.show()

### 5.4 Ozellik Pairplot (Secili Ozellikler)

In [ ]:
selected_features = ['mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'anomaly']
g = sns.pairplot(dataset[selected_features], hue='anomaly', palette={0: '#2ecc71', 1: '#e74c3c'},
                 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20})
g.fig.suptitle('Secili Ozellikler - Pairplot (Yesil: Normal, Kirmizi: Anomali)', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('../reports/figures/pairplot_ozellikler.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ⏰ Bolum 6: Zaman Serisi Analizi

### 6.1 Tum Kanallarin Zaman Serisi Gorsellestirmesi

In [ ]:
fig = make_subplots(rows=3, cols=3, subplot_titles=[CHANNEL_NAMES.get(c,c) for c in channels])

for i, ch in enumerate(channels):
    r, c = divmod(i, 3)
    ch_data = segments[segments['channel'] == ch].sort_values('timestamp')
    normal = ch_data[ch_data['anomaly'] == 0]
    anomaly = ch_data[ch_data['anomaly'] == 1]
    fig.add_trace(go.Scatter(x=normal['timestamp'], y=normal['value'],
                             mode='lines', name=f'{ch} Normal', line=dict(width=0.5),
                             showlegend=(i==0)), row=r+1, col=c+1)
    fig.add_trace(go.Scatter(x=anomaly['timestamp'], y=anomaly['value'],
                             mode='markers', name=f'{ch} Anomali',
                             marker=dict(size=3, color='red'), showlegend=(i==0)),
                  row=r+1, col=c+1)

fig.update_layout(title='📡 Tum Kanallarin Zaman Serisi (Normal vs Anomali)',
                  template='plotly_dark', height=900, showlegend=True)
fig.show()

### 6.2 Rolling Mean ve Rolling Std

In [ ]:
# Ornek kanal: CADC0872 (Manyetometre X)
sample_ch = 'CADC0872'
ch_data = segments[segments['channel'] == sample_ch].sort_values('timestamp').reset_index(drop=True)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Rolling Mean
axes[0].plot(ch_data.index, ch_data['value'], alpha=0.3, label='Ham Veri', color='gray')
for w, color in [(50, '#e74c3c'), (100, '#3498db'), (200, '#2ecc71')]:
    rm = ch_data['value'].rolling(window=w).mean()
    axes[0].plot(ch_data.index, rm, label=f'Rolling Mean (w={w})', linewidth=2, color=color)
axes[0].set_title(f'{CHANNEL_NAMES[sample_ch]} - Rolling Mean', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Deger')
axes[0].legend()

# Rolling Std
for w, color in [(50, '#e74c3c'), (100, '#3498db'), (200, '#2ecc71')]:
    rs = ch_data['value'].rolling(window=w).std()
    axes[1].plot(ch_data.index, rs, label=f'Rolling Std (w={w})', linewidth=2, color=color)
axes[1].set_title(f'{CHANNEL_NAMES[sample_ch]} - Rolling Std', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Ornek Indeksi')
axes[1].set_ylabel('Standart Sapma')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/rolling_stats.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Segment Bazinda Trend Analizi

In [ ]:
# Her segment icin ortalama deger trendi
seg_trend = segments.groupby(['channel', 'segment']).agg(
    mean_val=('value', 'mean'),
    std_val=('value', 'std'),
    count=('value', 'count'),
    anomaly=('anomaly', 'first')
).reset_index()

fig = px.scatter(seg_trend, x='segment', y='mean_val', color='anomaly',
                 facet_col='channel', facet_col_wrap=3,
                 title='📈 Segment Bazinda Ortalama Deger Trendi',
                 labels={'segment': 'Segment No', 'mean_val': 'Ortalama Deger', 'anomaly': 'Anomali'},
                 color_continuous_scale=['#2ecc71', '#e74c3c'])
fig.update_layout(template='plotly_dark', height=700)
fig.show()

---
## 🚨 Bolum 7: Anomali Etiketi Analizi

### 7.1 Anomali / Normal Sinif Dagilimi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
anomaly_counts = dataset['anomaly'].value_counts()
labels = ['Normal', 'Anomali']
colors = ['#2ecc71', '#e74c3c']
explode = (0, 0.05)
axes[0].pie(anomaly_counts, labels=labels, colors=colors, explode=explode,
            autopct='%1.1f%%', startangle=90, shadow=True, textprops={'fontsize': 13})
axes[0].set_title('Anomali/Normal Dagilimi (Pie)', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(labels, anomaly_counts.values, color=colors, edgecolor='black')
for i, v in enumerate(anomaly_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=13, fontweight='bold')
axes[1].set_title('Anomali/Normal Dagilimi (Bar)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Segment Sayisi')

plt.suptitle(f'📊 Toplam: {len(dataset)} segment | Normal: {anomaly_counts[0]} | Anomali: {anomaly_counts[1]}',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/anomali_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Kanal Bazinda Anomali Dagilimi

In [ ]:
ch_anomaly = dataset.groupby(['channel', 'anomaly']).size().reset_index(name='count')
ch_anomaly['label'] = ch_anomaly['anomaly'].map({0: 'Normal', 1: 'Anomali'})
ch_anomaly['sensor'] = ch_anomaly['channel'].map(CHANNEL_NAMES)

fig = px.bar(ch_anomaly, x='sensor', y='count', color='label', barmode='group',
             title='📡 Kanal Bazinda Normal vs Anomali Segment Sayisi',
             labels={'sensor': 'Sensor', 'count': 'Segment Sayisi'},
             color_discrete_map={'Normal': '#2ecc71', 'Anomali': '#e74c3c'})
fig.update_layout(template='plotly_dark', height=450)
fig.show()

### 7.3 Anomali Tipleri

In [ ]:
# Label dagilimi (anomali alt tipleri)
label_counts = segments[segments['anomaly'] == 1]['label'].value_counts()
print('=== Anomali Alt Tipleri ===')
display(pd.DataFrame({'Tip': label_counts.index, 'Sayi': label_counts.values,
                       'Oran (%)': (label_counts.values / label_counts.sum() * 100).round(1)}))

fig = px.pie(values=label_counts.values, names=label_counts.index,
             title='🏷️ Anomali Alt Tip Dagilimi',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(template='plotly_dark', height=400)
fig.show()

### 7.4 Zaman Ekseninde Anomali Konumlari

In [ ]:
fig = px.scatter(segments[segments['anomaly']==1], x='timestamp', y='value',
                 color='channel', symbol='label',
                 title='⏰ Zaman Ekseninde Anomali Konumlari',
                 labels={'timestamp': 'Zaman', 'value': 'Deger', 'channel': 'Kanal'},
                 opacity=0.6)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

---
## 📝 Bolum 8: Bulgular ve Ozet

### Temel Bulgular

1. **Veri Seti Yapisi:** Segments dosyasi 303,493 veri noktasi icerir, 9 farkli telemetri kanalindan toplanmistir.
2. **Zaman Araligi:** Ocak 2022 - Haziran 2022 (yaklasik 6 ay)
3. **Anomali Orani:** Dataset'te %20.4 anomali etiketi bulunmaktadir (434/2123 segment)
4. **Kanal Cesitliligi:** 3 Manyetometre + 6 Foto Diyot kanali
5. **Anomali Tipleri:** 'anomaly', 'a2', 'a3', 'a4' olmak uzere 4 farkli anomali alt tipi vardir

### Dikkat Edilmesi Gerekenler
- Bazi kanallarda (CADC0884) anomali etiketi bulunmamaktadir
- Veri noktasi sayisi kanallar arasinda buyuk farklilik gostermektedir
- Yuksek carpiklik ve basiklik degerleri bazi kanallarda gorulmektedir

### Sonraki Adimlar
- [ ] Veri on isleme (normalizasyon, eksik veri doldurma)
- [ ] Feature engineering (rolling, lag, diff ozellikleri)
- [ ] Supervised modeller (Random Forest, XGBoost, SVM)
- [ ] Unsupervised modeller (Isolation Forest, Autoencoder)
- [ ] Model karsilastirma ve degerlendirme

In [ ]:
print('='*60)
print('🛰️ EDA ANALIZI TAMAMLANDI')
print('='*60)
print(f'Toplam segment sayisi: {len(dataset):,}')
print(f'Toplam veri noktasi:   {len(segments):,}')
print(f'Kanal sayisi:          {segments["channel"].nunique()}')
print(f'Anomali segmentleri:   {(dataset["anomaly"]==1).sum()} ({(dataset["anomaly"]==1).mean()*100:.1f}%)')
print(f'Zaman araligi:         {segments["timestamp"].min()} - {segments["timestamp"].max()}')
print('='*60)

### 8.1 HTML Rapor Export

In [ ]:
# HTML Rapor Olusturma
!jupyter nbconvert --to html 01_veri_inceleme.ipynb --output ../reports/01_veri_inceleme_rapor.html
print("HTML Raporu reports/01_veri_inceleme_rapor.html konumuna kaydedildi.")